# 15. RAG Agent 進階版

使用向量資料庫和 ReAct 模式建立完整的 RAG Agent。

---

## 🎯 學習目標

- ✅ 使用 FAISS 向量資料庫
- ✅ 實作 Self-RAG 模式（自我評估）
- ✅ 整合 ReAct 迴圈
- ✅ 文件相關性評估

---

## 📊 Self-RAG 架構

```
┌─────────────────────────────────────────────────────────────┐
│                    Self-RAG + ReAct 架構                     │
├─────────────────────────────────────────────────────────────┤
│                                                              │
│   ┌──────────┐     ┌──────────┐     ┌──────────┐            │
│   │   問題   │ ──▶ │  檢索器  │ ──▶ │ 評估器   │            │
│   └──────────┘     └──────────┘     └────┬─────┘            │
│                                          │                   │
│                           ┌──────────────┼──────────────┐   │
│                           │              │              │   │
│                           ▼              ▼              ▼   │
│                      ┌────────┐    ┌────────┐    ┌────────┐│
│                      │ 相關   │    │ 不確定 │    │ 不相關 ││
│                      └───┬────┘    └───┬────┘    └───┬────┘│
│                          │             │             │      │
│                          ▼             ▼             ▼      │
│                      ┌────────┐    ┌────────┐    ┌────────┐│
│                      │ 生成   │    │ 改寫問題│    │ 網路搜尋││
│                      └───┬────┘    └───┬────┘    └────────┘│
│                          │             │                    │
│                          ▼             ▼                    │
│                      ┌────────┐    回到檢索                 │
│                      │ 幻覺檢查│                             │
│                      └───┬────┘                             │
│                          │                                   │
│                          ▼                                   │
│                      ┌────────┐                             │
│                      │  回答  │                             │
│                      └────────┘                             │
│                                                              │
└─────────────────────────────────────────────────────────────┘
```

In [1]:
from typing import TypedDict, Annotated, Literal
from langgraph.graph import StateGraph, START, END
import re
from pathlib import Path
import os
from dotenv import load_dotenv
load_dotenv()

True

---

## 15.1 向量檢索器

使用簡易的餘弦相似度（無需外部套件）：

In [2]:
import math
from collections import Counter

def text_to_vector(text: str) -> dict:
    """將文本轉為詞頻向量（簡易版 TF）"""
    words = re.findall(r'\w+', text.lower())
    return Counter(words)

def cosine_similarity(vec1: dict, vec2: dict) -> float:
    """計算餘弦相似度"""
    # 找共同的詞
    intersection = set(vec1.keys()) & set(vec2.keys())
    
    # 計算點積
    dot_product = sum(vec1[word] * vec2[word] for word in intersection)
    
    # 計算模長
    mag1 = math.sqrt(sum(v**2 for v in vec1.values()))
    mag2 = math.sqrt(sum(v**2 for v in vec2.values()))
    
    if mag1 == 0 or mag2 == 0:
        return 0.0
    
    return dot_product / (mag1 * mag2)

print("✅ 向量檢索工具已定義")

✅ 向量檢索工具已定義


---

## 15.2 載入並建立索引

In [3]:
# 載入知識庫
knowledge_path = Path("../data/langgraph_knowledge.md")

if knowledge_path.exists():
    with open(knowledge_path, "r", encoding="utf-8") as f:
        raw_content = f.read()
else:
    raw_content = "LangGraph 是一個用於構建有狀態應用程式的框架。"

# 切分文件
def split_document(text: str, chunk_size: int = 400) -> list[dict]:
    paragraphs = re.split(r'\n\n+', text)
    chunks = []
    current = ""
    
    for para in paragraphs:
        if len(current) + len(para) < chunk_size:
            current += para + "\n\n"
        else:
            if current:
                chunks.append(current.strip())
            current = para + "\n\n"
    if current:
        chunks.append(current.strip())
    
    return [{"id": i, "content": c, "vector": text_to_vector(c)} for i, c in enumerate(chunks)]

# 建立索引
document_index = split_document(raw_content)
print(f"✅ 索引建立完成: {len(document_index)} 個區塊")

✅ 索引建立完成: 13 個區塊


In [4]:
def vector_search(query: str, top_k: int = 3) -> list[dict]:
    """向量搜尋"""
    query_vec = text_to_vector(query)
    
    results = []
    for doc in document_index:
        score = cosine_similarity(query_vec, doc["vector"])
        if score > 0:
            results.append({
                "id": doc["id"],
                "content": doc["content"],
                "score": round(score, 4)
            })
    
    results.sort(key=lambda x: x["score"], reverse=True)
    return results[:top_k]

# 測試
print("📊 測試向量搜尋:")
for r in vector_search("如何使用 StateGraph"):
    print(f"  [相似度: {r['score']}] {r['content'][:50]}...")

📊 測試向量搜尋:
  [相似度: 0.2287] # LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph ...
  [相似度: 0.0962] - `"updates"`: 只返回更新的欄位
- `"values"`: 返回完整狀態快照

##...
  [相似度: 0.0845] class MyState(TypedDict):
    messages: list
    c...


---

## 15.3 定義 Self-RAG 狀態

In [5]:
class SelfRAGState(TypedDict):
    """Self-RAG Agent 狀態"""
    question: str              # 原始問題
    rewritten_question: str    # 改寫後的問題
    documents: list            # 檢索到的文件
    relevance: str             # 相關性: relevant / not_relevant / uncertain
    generation: str            # 生成的回答
    is_hallucination: bool     # 是否幻覺
    retry_count: int           # 重試次數
    max_retries: int           # 最大重試

print("✅ Self-RAG 狀態定義完成")

✅ Self-RAG 狀態定義完成


---

## 15.4 定義 Self-RAG 節點

In [6]:
def retrieve_documents(state: SelfRAGState) -> dict:
    """檢索相關文件"""
    query = state.get("rewritten_question") or state["question"]
    print(f"  📚 檢索: {query[:30]}...")
    
    docs = vector_search(query, top_k=3)
    print(f"  📄 找到 {len(docs)} 個文件 (最高分: {docs[0]['score'] if docs else 0})")
    
    return {"documents": docs}

def grade_documents(state: SelfRAGState) -> dict:
    """評估文件相關性"""
    docs = state.get("documents", [])
    
    if not docs:
        print("  ❌ 無文件，標記為不相關")
        return {"relevance": "not_relevant"}
    
    # 根據分數判斷相關性
    top_score = docs[0]["score"]
    
    if top_score > 0.3:
        print(f"  ✅ 高相關性 (score: {top_score})")
        return {"relevance": "relevant"}
    elif top_score > 0.15:
        print(f"  🔶 不確定 (score: {top_score})")
        return {"relevance": "uncertain"}
    else:
        print(f"  ❌ 低相關性 (score: {top_score})")
        return {"relevance": "not_relevant"}

def rewrite_question(state: SelfRAGState) -> dict:
    """改寫問題以改善檢索"""
    original = state["question"]
    retry = state.get("retry_count", 0) + 1
    
    # 簡單的問題改寫（實際應用中會用 LLM）
    rewrites = {
        1: f"{original} 定義 教學 範例",
        2: f"{original} LangGraph 用法",
    }
    
    rewritten = rewrites.get(retry, original)
    print(f"  ✏️ 改寫問題 (第{retry}次): {rewritten[:40]}...")
    
    return {"rewritten_question": rewritten, "retry_count": retry}

def generate_response(state: SelfRAGState) -> dict:
    """生成回答"""
    docs = state.get("documents", [])
    question = state["question"]
    
    print("  🤖 生成回答...")
    
    if docs:
        context = "\n\n".join([d["content"][:150] for d in docs[:2]])
        answer = f"關於『{question}』，根據知識庫：\n\n{context}\n\n(基於 {len(docs)} 個相關文件)"
    else:
        answer = f"抱歉，無法找到與『{question}』相關的資訊。"
    
    return {"generation": answer}

def check_hallucination(state: SelfRAGState) -> dict:
    """檢查是否幻覺（簡化版）"""
    docs = state.get("documents", [])
    generation = state.get("generation", "")
    
    # 簡單檢查：回答是否包含文件中的關鍵內容
    if not docs:
        return {"is_hallucination": False}  # 無文件時不算幻覺
    
    # 檢查是否有文件內容在回答中
    has_grounding = any(
        any(word in generation.lower() for word in doc["content"].lower().split()[:10])
        for doc in docs
    )
    
    print(f"  🔍 幻覺檢查: {'通過' if has_grounding else '可能幻覺'}")
    return {"is_hallucination": not has_grounding}

def fallback_answer(state: SelfRAGState) -> dict:
    """備用回答"""
    print("  ⚠️ 使用備用回答")
    return {"generation": f"關於『{state['question']}』，建議參考官方文件或嘗試更具體的問題。"}

print("✅ Self-RAG 節點定義完成")

✅ Self-RAG 節點定義完成


---

## 15.5 建構 Self-RAG 圖

In [7]:
def route_by_relevance(state: SelfRAGState) -> Literal["generate", "rewrite", "fallback"]:
    """根據相關性路由"""
    relevance = state.get("relevance", "not_relevant")
    retry = state.get("retry_count", 0)
    max_retries = state.get("max_retries", 2)
    
    if relevance == "relevant":
        return "generate"
    elif retry < max_retries:
        return "rewrite"
    else:
        return "fallback"

def route_by_hallucination(state: SelfRAGState) -> Literal["done", "retry"]:
    """根據幻覺檢查結果路由"""
    if state.get("is_hallucination") and state.get("retry_count", 0) < state.get("max_retries", 2):
        return "retry"
    return "done"

# 建構圖
graph = StateGraph(SelfRAGState)

# 添加節點
graph.add_node("retrieve", retrieve_documents)
graph.add_node("grade", grade_documents)
graph.add_node("rewrite", rewrite_question)
graph.add_node("generate", generate_response)
graph.add_node("hallucination_check", check_hallucination)
graph.add_node("fallback", fallback_answer)

# 添加邊
graph.add_edge(START, "retrieve")
graph.add_edge("retrieve", "grade")
graph.add_conditional_edges("grade", route_by_relevance, {
    "generate": "generate",
    "rewrite": "rewrite",
    "fallback": "fallback"
})
graph.add_edge("rewrite", "retrieve")  # 改寫後重新檢索
graph.add_edge("generate", "hallucination_check")
graph.add_conditional_edges("hallucination_check", route_by_hallucination, {
    "done": END,
    "retry": "rewrite"
})
graph.add_edge("fallback", END)

# 編譯
self_rag_app = graph.compile()
print("✅ Self-RAG Agent 已就緒")

✅ Self-RAG Agent 已就緒


In [8]:
print("📊 Self-RAG 圖結構:")
print(self_rag_app.get_graph().draw_mermaid())

📊 Self-RAG 圖結構:
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade(grade)
	rewrite(rewrite)
	generate(generate)
	hallucination_check(hallucination_check)
	fallback(fallback)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	generate --> hallucination_check;
	grade -.-> fallback;
	grade -.-> generate;
	grade -.-> rewrite;
	hallucination_check -. &nbsp;done&nbsp; .-> __end__;
	hallucination_check -. &nbsp;retry&nbsp; .-> rewrite;
	retrieve --> grade;
	rewrite --> retrieve;
	fallback --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



---

## 15.6 測試 Self-RAG Agent

In [9]:
print("🚀 測試 Self-RAG Agent:")
print("=" * 70)

questions = [
    "什麼是 StateGraph？",
    "Reducer 如何使用？",
    "量子計算原理",  # 知識庫中沒有
]

for q in questions:
    print(f"\n❓ 問題: {q}")
    print("-" * 50)
    
    result = self_rag_app.invoke({
        "question": q,
        "rewritten_question": "",
        "documents": [],
        "relevance": "",
        "generation": "",
        "is_hallucination": False,
        "retry_count": 0,
        "max_retries": 2
    })
    
    print(f"\n💡 回答:\n{result['generation'][:200]}...")
    print(f"\n📊 統計: 重試 {result['retry_count']} 次")
    print("\n" + "=" * 70)

🚀 測試 Self-RAG Agent:

❓ 問題: 什麼是 StateGraph？
--------------------------------------------------
  📚 檢索: 什麼是 StateGraph？...
  📄 找到 3 個文件 (最高分: 0.305)
  ✅ 高相關性 (score: 0.305)
  🤖 生成回答...
  🔍 幻覺檢查: 通過

💡 回答:
關於『什麼是 StateGraph？』，根據知識庫：

# LangGraph 核心概念知識庫

## 什麼是 LangGraph？

LangGraph 是一個用於構建有狀態、多角色應用程式的框架，專為 LLM（大型語言模型）設計。它擴展了 LangChain，提供了循環計算和狀態管理的能力。

### 核心特色

1. **循環支持**：與 DAG（

- `"updates"`: 只返回更新...

📊 統計: 重試 0 次


❓ 問題: Reducer 如何使用？
--------------------------------------------------
  📚 檢索: Reducer 如何使用？...
  📄 找到 2 個文件 (最高分: 0.3592)
  ✅ 高相關性 (score: 0.3592)
  🤖 生成回答...
  🔍 幻覺檢查: 通過

💡 回答:
關於『Reducer 如何使用？』，根據知識庫：

邊定義節點之間的連接。有兩種類型：
1. **普通邊**：`add_edge(from, to)` - 固定路徑
2. **條件邊**：`add_conditional_edges(from, router, path_map)` - 動態路徑

## Reducer 機制

Reducer 決定如

- `"updates"`: 只返回更新的欄...

📊 統計: 重試 0 次


❓ 問題: 量子計算原理
--------------------------------------------------
  📚 檢索: 量子計算原理...
  📄 找到 0 個文件 (最高分: 0)
  ❌ 無文件，標記為不相關
  ✏️ 改寫問題 (第1次): 量子計算原理 定義 教學 範例...
  📚 檢索: 量子計算原理 定義 教學 

---

## 💡 重點回顧

### Self-RAG 特點

| 特點 | 說明 |
|------|------|
| 自我評估 | 評估檢索文件相關性 |
| 問題改寫 | 當結果不佳時自動改寫 |
| 幻覺檢查 | 驗證回答是否基於文件 |
| 重試機制 | 有限次數重試 |

### 進階應用

1. **真實向量庫**: 使用 FAISS/Chroma/Pinecone
2. **真實 LLM**: 用 GPT-4 評估相關性和生成
3. **多源檢索**: 結合網路搜尋

---

🎉 恭喜完成 RAG Agent 系列！